In [2]:
!pip install -q google-genai pypdf

from google import genai
from google.genai import types
import numpy as np
from google.colab import userdata

client=genai.Client(api_key=userdata.get('Ragproject'))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 8.8 MB/s eta 0:00:00


In [3]:
from google.colab import files

upload = files.upload()

pdf_name=list(upload.keys())[0]
print("Uploaded pdf file: ",pdf_name)

Saving AquaPure_RO300_Manual.pdf to AquaPure_RO300_Manual.pdf
Uploaded pdf file:  AquaPure_RO300_Manual.pdf


In [6]:
from pypdf import PdfReader

read = PdfReader(pdf_name)
print("Total number of pages in the pdf: ",len(read.pages))

text=''
for pg in read.pages:
  text+=pg.extract_text() + '\n'

print("Total number of characters in the pdf: ",len(text))

Total number of pages in the pdf:  3
Total number of characters in the pdf:  2818


In [32]:
def chunking(text,chunk_size=800,overlap=150):
  chunk=[]
  start=0
  while(start<len(text)):
    end=start+chunk_size
    chunk.append(text[start:end])
    start=end-overlap
  return chunk

chunks=chunking(text)
print("Total number of chunks:",len(chunks))

Total number of chunks: 5


In [33]:
print(chunks[0][-150:])
print()
print(chunks[1][:150])

imum inlet water pressure is 5 psi. Connect the inlet pipe to
the tap using the supplied connector and open the inlet valve slowly. Do not install the

imum inlet water pressure is 5 psi. Connect the inlet pipe to
the tap using the supplied connector and open the inlet valve slowly. Do not install the


In [34]:
EMBEDDING_MODEL = 'gemini-embedding-001'
EMBEDDING_DIMENSION=768

def embedding(t):
  response=client.models.embed_content(
      model=EMBEDDING_MODEL,
      contents=t,
      config=types.EmbedContentConfig(
          output_dimensionality=EMBEDDING_DIMENSION
      )
  )
  return(response.embeddings[0].values)

embed=[]
for i, ch in enumerate(chunks):
  embed.append(np.array(embedding(ch)))
  print(f"chunk {i+1}/{len(chunks)} done")

print()
chunk_vector=np.array(embed)
print("Chunk vector shape: ",chunk_vector.shape)

print()
chunk_vector=chunk_vector/np.linalg.norm(chunk_vector,axis=1,keepdims=True)
print("Normalized Chunk vector shape: ",chunk_vector.shape)

chunk 1/5 done
chunk 2/5 done
chunk 3/5 done
chunk 4/5 done
chunk 5/5 done

Chunk vector shape:  (5, 768)

Normalized Chunk vector shape:  (5, 768)


In [35]:
def retrieve(question, top_k=3):

    q_vector = np.array(embedding(question))
    q_vector = q_vector / np.linalg.norm(q_vector)

    score = chunk_vector @ q_vector

    top_result = np.argsort(score)[::-1][:top_k]

    return [(chunks[i], score[i]) for i in top_result]

In [36]:
result = retrieve("What does error E2 mean?")

for chunk, score in result:
  print(f"Score: {score:.2f}")
  print(chunk)
  print()

Score: 0.63
o input water supply. Check that the inlet valve is open and the tap has water. E2: UV lamp failure.
The UV lamp must be replaced by a service technician. E3: Tank full sensor error. Switch off the purifier
for 5 minutes and switch on again. E4: Water leak detected. Switch off the power immediately, close the
inlet valve, and call customer care.
5. Troubleshooting
If no water comes out of the tap, first check whether the display shows E1. If the water tastes bad, the
filters may need replacement. If the purifier makes a loud noise, check that the inlet water pressure is
not below 5 psi and that the unit is fixed firmly on the wall.

6. Filter Replacement Schedule
The sediment filter must be replaced every 6 months. The carbon filter must be replaced every 6
months. The RO membrane must be 

Score: 0.59
imum inlet water pressure is 5 psi. Connect the inlet pipe to
the tap using the supplied connector and open the inlet valve slowly. Do not install the purifier in direct
sunl

In [42]:
configuration = types.GenerateContentConfig(
    system_instruction=(
        "You are ManualMate, an assistant that answers questions about the AquaPure RO-300 water purifier using only the manual excerpts provided in the context. "
        "Each excerpt is labelled with its page number like [Page 2]. "
        "End every answer with the page number(s) you used, like (Page 2). "
        "Give steps in simple words. "
        "If the answer is not in the context, reply exactly: "
        "I could not find this in the manual. "
        "Never guess or use outside knowledge."
    ),
    temperature=0.3
)

def ask(question, top_k=3):
  top = retrieve(question,top_k)

  content = '\n\n_______\n\n'.join(chunk for chunk, score in top)

  prompt = f"""Manual excerpts:
          {content}
          QUESTION: {question}
          Answer only from the excerpts above and mention the page number."""

  response=client.models.generate_content(
      model='gemini-3.1-flash-lite',
      contents=prompt,
      config=configuration
  )
  return response.text

print(ask('What is the storage tank capacity?'))

The storage tank capacity is 8 litres. (Page 1)


In [43]:
print(ask('How often should the RO membrane be replaced?'))

The RO membrane must be replaced every 18 months. (Page 1)


In [44]:
print(ask('What is the price of this purifier?'))

I could not find this in the manual.
